# Callback mining: Reward + Loss (Colab + Gradio)

Runs in **Google Colab** after upload.
- DQN on **CartPole-v1**
- **RewardAndLossCallback**: collect episode reward + `train/loss`
- Use SB3 `model.logger.name_to_value['train/loss']` for charts
- Gradio: learning curve (reward + Loss) and summary text
- **DQN analysis (heatmap)**: Q-value and policy (action) heatmaps after training
- **GPU/CPU supported** (CUDA if GPU, else CPU)

## 1. Install libraries

In [ ]:
!pip install -q gymnasium[classic_control] stable-baselines3 gradio matplotlib

## 2. Callback reward/Loss collection and DQN training

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")

import numpy as np
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr

# Reuse trained model for DQN analysis (heatmap)
_last_model = None


def get_device():
    """GPU and CPU supported: CUDA > MPS > CPU. Works on Colab GPU/CPU runtime."""
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


class RewardAndLossCallback(BaseCallback):
    """Callback that collects episode reward + train/loss for live charts."""

    def __init__(self, reward_list, step_list, loss_list, loss_step_list, verbose=0):
        super().__init__(verbose)
        self.reward_list = reward_list
        self.step_list = step_list
        self.loss_list = loss_list
        self.loss_step_list = loss_step_list
        self._episode_reward = 0.0

    def _on_step(self):
        self._episode_reward += self.locals["rewards"][0]
        if self.locals["dones"][0]:
            self.reward_list.append(self._episode_reward)
            self.step_list.append(self.num_timesteps)
            self._episode_reward = 0.0

        if self.model.logger is not None:
            name_to_val = getattr(self.model.logger, "name_to_value", None)
            if name_to_val and "train/loss" in name_to_val:
                loss = name_to_val["train/loss"]
                self.loss_list.append(float(loss))
                self.loss_step_list.append(self.num_timesteps)
        return True


def run_callback_reward_loss(
    total_timesteps=30000,
    log_interval=100,
    learning_rate=1e-3,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=32,
    gamma=0.99,
    target_update_interval=1_000,
    train_freq=4,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
):
    env = gym.make("CartPole-v1")
    device = get_device()
    device_name = "CUDA" if device == "cuda" else ("MPS" if device == "mps" else "CPU")

    reward_list = []
    step_list = []
    loss_list = []
    loss_step_list = []
    callback = RewardAndLossCallback(
        reward_list, step_list, loss_list, loss_step_list
    )

    model = DQN(
        policy="MlpPolicy",
        env=env,
        device=device,
        learning_rate=float(learning_rate),
        buffer_size=int(buffer_size),
        learning_starts=int(learning_starts),
        batch_size=int(batch_size),
        gamma=float(gamma),
        target_update_interval=int(target_update_interval),
        train_freq=int(train_freq),
        exploration_fraction=float(exploration_fraction),
        exploration_initial_eps=float(exploration_initial_eps),
        exploration_final_eps=float(exploration_final_eps),
        verbose=1,
    )
    model.learn(
        total_timesteps=total_timesteps,
        callback=callback,
        log_interval=log_interval,
    )
    env.close()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=False)

    if step_list and reward_list:
        ax1.plot(step_list, reward_list, alpha=0.4, color="steelblue", label="Episode reward")
        if len(reward_list) >= 10:
            window = min(30, len(reward_list) // 5)
            kernel = np.ones(window) / window
            smoothed = np.convolve(reward_list, kernel, mode="same")
            ax1.plot(step_list, smoothed, color="coral", linewidth=2, label=f"Moving avg (w={window})")
        ax1.set_ylabel("Episode reward")
        ax1.set_title("Reward from callback")
        ax1.legend(loc="lower right")
        ax1.grid(True, alpha=0.3)

    if loss_step_list and loss_list:
        ax2.plot(loss_step_list, loss_list, alpha=0.6, color="green", linewidth=0.8, label="train/loss")
        ax2.set_xlabel("Step")
        ax2.set_ylabel("train/loss")
        ax2.set_title("Loss from callback (model.logger.name_to_value['train/loss'])")
        ax2.legend(loc="upper right")
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "Loss appears in log after training runs.", ha="center", va="center")
        ax2.set_xlabel("Step")

    plt.tight_layout()

    txt = f"""Device: {device_name}
Episode rewards collected: {len(reward_list)}
Loss samples collected: {len(loss_list)}
total_timesteps: {total_timesteps}, log_interval: {log_interval}
lr: {learning_rate}, buffer: {buffer_size}, batch: {batch_size}, gamma: {gamma}
exploration: {exploration_fraction} (ε {exploration_initial_eps}→{exploration_final_eps})"""
    global _last_model
    _last_model = model
    return fig, txt


def run_dqn_heatmap():
    """Draw Q-value and policy heatmaps from trained DQN."""
    global _last_model
    if _last_model is None:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.text(0.5, 0.5, "Run 'Train and show results' first.", ha="center", va="center", fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        return fig, "No trained model. Run training above first then try again."

    import torch
    # CartPole-v1: obs = [pos, vel, angle, angle_vel]. 2D (pos, angle) grid, rest 0
    pos_min, pos_max = -2.4, 2.4
    th_min, th_max = -0.3, 0.3
    n = 40
    positions = np.linspace(pos_min, pos_max, n)
    angles = np.linspace(th_min, th_max, n)
    Q_left = np.zeros((n, n))
    Q_right = np.zeros((n, n))
    actions_grid = np.zeros((n, n), dtype=int)
    for i, pos in enumerate(positions):
        for j, th in enumerate(angles):
            obs = np.array([[pos, 0.0, th, 0.0]], dtype=np.float32)
            with torch.no_grad():
                obs_t = torch.as_tensor(obs).to(_last_model.device)
                q_vals = _last_model.policy.q_net(obs_t)
            q = q_vals.cpu().numpy().squeeze()
            Q_left[i, j] = q[0]
            Q_right[i, j] = q[1]
            actions_grid[i, j] = 0 if q[0] >= q[1] else 1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    maxQ = np.maximum(Q_left, Q_right)
    im1 = ax1.imshow(maxQ.T, origin="lower", extent=[pos_min, pos_max, th_min, th_max], aspect="auto", cmap="viridis")
    ax1.set_xlabel("Position")
    ax1.set_ylabel("Angle (rad)")
    ax1.set_title("Q-value (max_a Q(s,a))")
    plt.colorbar(im1, ax=ax1)
    im2 = ax2.imshow(actions_grid.T, origin="lower", extent=[pos_min, pos_max, th_min, th_max], aspect="auto", cmap="coolwarm", vmin=0, vmax=1)
    ax2.set_xlabel("Position")
    ax2.set_ylabel("Angle (rad)")
    ax2.set_title("Action (0=left, 1=right)")
    plt.colorbar(im2, ax=ax2, ticks=[0, 1])
    plt.tight_layout()
    txt = "CartPole 2D slice: position-angle grid (vel=0, angle_vel=0). Q-value heatmap + policy (action) heatmap."
    return fig, txt

## 3. Run Gradio app

### Callback / DQN parameter guide

| Parameter | Description |
|----------|------|
| **total_timesteps** | Total env steps for DQN training |
| **log_interval** | SB3 log interval (affects train/loss recording) |
| **learning_rate** | Adam learning rate |
| **buffer_size** | Experience replay buffer size |
| **learning_starts** | Steps to collect before training starts |
| **batch_size** | Minibatch size |
| **gamma** | Discount factor |
| **target_update_interval** | Target network update period (steps) |
| **train_freq** | Train every N steps |
| **exploration_fraction** | Fraction of training for epsilon decay |
| **exploration_initial_eps / final_eps** | epsilon-greedy initial/final value |

### DQN analysis (heatmap)

With a trained DQN you can plot:
- **Q-value heatmap**: max_a Q(s,a) over 2D state grid (position, angle).
- **Policy (action) heatmap**: action chosen by argmax Q(s,a) (0=left, 1=right).
Run training first, then click **DQN analysis (heatmap)** to draw both heatmaps from the last model.

In [ ]:
with gr.Blocks(title="Callback Reward+Loss - CartPole", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Callback mining: Reward + Loss")
    gr.Markdown("CartPole-v1 DQN. Callback collects episode reward and train/loss for charts.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Training scale")
            total_timesteps = gr.Slider(5000, 100000, value=30000, step=5000, label="total_timesteps")
            log_interval = gr.Slider(10, 500, value=100, step=10, label="log_interval")

            gr.Markdown("### Experience replay")
            buffer_size = gr.Slider(5000, 100000, value=50_000, step=5000, label="buffer_size")
            learning_starts = gr.Slider(500, 10000, value=1_000, step=500, label="learning_starts")
            batch_size = gr.Slider(16, 256, value=32, step=16, label="batch_size")
            train_freq = gr.Slider(1, 16, value=4, step=1, label="train_freq")

            gr.Markdown("### Target network / General")
            target_update_interval = gr.Slider(100, 5000, value=1_000, step=100, label="target_update_interval")
            learning_rate = gr.Dropdown(choices=[1e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2], value=1e-3, label="learning_rate")
            gamma = gr.Slider(0.9, 1.0, value=0.99, step=0.01, label="gamma")

            gr.Markdown("### Exploration (epsilon-greedy)")
            exploration_fraction = gr.Slider(0.05, 0.5, value=0.1, step=0.05, label="exploration_fraction")
            exploration_initial_eps = gr.Slider(0.5, 1.0, value=1.0, step=0.05, label="exploration_initial_eps")
            exploration_final_eps = gr.Slider(0.0, 0.2, value=0.05, step=0.01, label="exploration_final_eps")

            run_btn = gr.Button("Train and show results", variant="primary")

        with gr.Column(scale=2):
            plot_out = gr.Plot(label="Reward + Loss curve")
            text_out = gr.Textbox(label="Results", lines=8, interactive=False)

    def run_fn(ts, log_int, lr, buf, lstart, batch, tfreq, target_int, gam, exp_frac, exp_init, exp_final):
        return run_callback_reward_loss(
            total_timesteps=int(ts),
            log_interval=int(log_int),
            learning_rate=lr,
            buffer_size=int(buf),
            learning_starts=int(lstart),
            batch_size=int(batch),
            train_freq=int(tfreq),
            target_update_interval=int(target_int),
            gamma=float(gam),
            exploration_fraction=float(exp_frac),
            exploration_initial_eps=float(exp_init),
            exploration_final_eps=float(exp_final),
        )

    run_btn.click(
        fn=run_fn,
        inputs=[
            total_timesteps, log_interval,
            learning_rate, buffer_size, learning_starts, batch_size, train_freq,
            target_update_interval, gamma,
            exploration_fraction, exploration_initial_eps, exploration_final_eps,
        ],
        outputs=[plot_out, text_out],
    )

    gr.Markdown("---\n### DQN analysis (heatmap)\nAfter training, use the button below to plot Q-value and policy heatmaps.")
    heatmap_btn = gr.Button("DQN analysis (heatmap)", variant="secondary")
    plot_heat = gr.Plot(label="Q-value / Policy heatmap")
    text_heat = gr.Textbox(label="Heatmap description", lines=3)
    heatmap_btn.click(fn=run_dqn_heatmap, outputs=[plot_heat, text_heat])

demo.launch(share=False)  # share=True for public URL